# Lab 1 · Fourier Neural Operator for the 1D Burgers equation — **30 minutes**

**LETMA 2026 Summer School · Lab 1 of 3** — Lab 2: FFNO for infrasound propagation · Lab 3: Graph Neural Networks.

**One scientific question.** *Can a network learn the solution operator of a PDE — the map from an
initial condition to the solution at a later time — and what do the training loss and its
hyperparameters do to what is learned?*

| Time | Section | What you will see |
|---|---|---|
| 00–02 | §0 Setup | device, imports |
| 02–08 | §1 The problem, stated with every hypothesis | Burgers dynamics animated; the dataset |
| 08–14 | §2 The one new idea: spectral convolution | modes, truncation, **Exercise F1 (5 min)** |
| 14–20 | §3 Assemble the FNO and train it | **animation of the prediction improving while training** |
| 20–26 | §4 The loss, its hyperparameters, its landscape | sweep + **animated optimizer race to the final loss** |
| 26–30 | §5 Evaluate, super-resolve, transfer | test error, zero-shot super-resolution, checklist |

**🥾 Optional deep dives** (skip them live; their outputs are all in the executed solution):
§4.5 the landscape in 3D, §4.6 the hyperparameter corner plot.
**How to finish in 30 minutes — read this before starting.** The critical path is the table
above; the ⏱ checkpoints at each section header tell you where you should be. Three safety
mechanisms keep everyone on track, whatever your ML background:

1. **Exercises cannot block you.** Each one has an automatic checker; if your attempt does not
   pass, the reference implementation takes over and the lab continues — a failed exercise
   costs only the minutes you spent on it, never the lab.
2. **🥾 Optional deep dives are skippable** without losing the thread: all their outputs are
   in the executed solution notebook, and the instructor shows them live if time allows.
3. **🧭 boxes are the novice path.** If you are new to ML: read the 🧭 boxes, watch the
   animations, do the exercise, and keep moving. The 📐 boxes are reference material to
   revisit afterwards — nobody is expected to absorb every formula live.

All the training on the critical path together takes seconds on the cluster GPU and under
~2 min on a laptop CPU: the clock is spent on reading and watching, not on waiting.


**How to use this notebook.** 🧭 boxes assume **no machine-learning background**: every term is
defined before it is used. 📐 boxes are for the **rigorous reader**: every assumption is stated as a
numbered hypothesis (H1, H2, …) and used nowhere else. The notebook runs on the Hedy cluster **or on
any laptop CPU**: no internet, no GPU, no pre-trained model. Run the cells in order; animations are
inline JavaScript, press ▶ inside each widget.

## §0 · Setup — 00–02 min · ⏱ running by minute 2

Run the two cells below. Check the reported device (`cpu` or `cuda`): everything that follows adapts
automatically. On Hedy, `LAB_PLOT_MODE=tk` before launching the kernel gives interactive X11 plots;
the default `inline` mode embeds everything in the notebook.

---
**Authors:** E. Noëlé¹²³⁴, J.-P. Gatti⁵, L. Lehmann⁵, C. Millet⁵, D. Clouteau³

**Affiliations:**
¹ AMIAD — ² CEA — ³ LMPS (Laboratoire de Mécanique Paris-Saclay, CNRS/ENS
Paris-Saclay/CentraleSupélec/Université Paris-Saclay) — ⁴ Centre Borelli (ENS Paris-Saclay) —
⁵ CEA/DAM

**LETMA 2026 Summer School** · Machine Learning on Geophysical Data


In [ ]:
# Portable setup: runs unchanged on the Hedy cluster or on a laptop. Run this cell first.
import os, sys
from pathlib import Path
project_root = next((p for p in [Path(os.environ.get('HEDY_PROJECT_ROOT', Path.cwd())),
                                  Path.cwd(), *Path.cwd().parents]
                     if (p / 'fno_tuto').is_dir() and (p / 'gnn_tuto').is_dir()), None)
if project_root is None:
    raise RuntimeError('Open this notebook inside the tutorial project directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# LAB_PLOT_MODE=tk only for interactive SSH/X11 on Hedy (set before launching the kernel).
from IPython import get_ipython
PLOT_MODE = os.environ.get('LAB_PLOT_MODE', 'inline').strip().lower()
get_ipython().run_line_magic('matplotlib', PLOT_MODE)

import math, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

from fno_tuto.plotting import setup_style
setup_style()
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.titlesize': 12,
                     'axes.labelsize': 11, 'legend.fontsize': 10})
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)

# Optional Hedy session profiling: active only on the cluster, silently skipped elsewhere.
HEDY_SESSION = None
try:
    from hedy_euporie.profiler import begin_session
    HEDY_SESSION = begin_session('lab_fno_student', backend='torch')
except Exception:
    print('Hedy session profiling not available here (local/laptop run): continuing without it.')

def show_anim(fig, func, frames, interval=200, blit=False):
    """Inline JavaScript animation: works in Jupyter, JupyterLab and Euporie."""
    ani = animation.FuncAnimation(fig, func, frames=frames, interval=interval, blit=blit)
    plt.close(fig)          # suppress the empty static frame
    display(HTML(ani.to_jshtml()))

print(f'Python {sys.version.split()[0]} | torch {torch.__version__} | device {DEVICE}')

In [ ]:
from fno_tuto.data import load_burgers, solve_burgers
from fno_tuto.models import FNO1d, SpectralConv1d
from fno_tuto.losses import get_loss, all_metrics
from fno_tuto.training import ChannelNormalizer
from fno_tuto.td import check_spectral_conv1d
print('imports OK')

## §1 · The problem, stated with every hypothesis — 02–08 min · ⏱ checkpoint: start §2 at minute 8

We approximate the **time-$T$ map** of the viscous 1D Burgers equation on the periodic domain
$[0,1)$:

$$\partial_t u(x,t) + u\,\partial_x u(x,t) = \nu\, \partial_{xx} u(x,t),
\qquad u(x,0)=u_0(x), \qquad \nu = 0.008,\; T = 0.3 .$$

The nonlinear term steepens the wave (it transports crests faster than troughs), the viscous term
smooths it: the two fight each other, and $u(\cdot,T)$ is a **nonlinear, nonlocal functional of
$u_0$**. That is the object we want to learn: the operator

$$G^\dagger: u_0 \longmapsto u(\cdot, T), \qquad G^\dagger: L^2_{\mathrm{per}} \to L^2_{\mathrm{per}}.$$

**Every hypothesis we make, explicitly.** Nothing outside this list is assumed anywhere in the lab.

- **H1 (data distribution).** Initial conditions are i.i.d. samples of a periodic Gaussian random
  field with Matérn-type spectrum $|\hat u_0(k)| \propto (4\pi^2 k^2 + \tau^2)^{-\alpha/2}$,
  $\alpha = 2.5$, $\tau = 12$, normalized to unit variance. Smooth, few big bumps: no white noise,
  no shocks at $t=0$.
- **H2 (fixed dynamics).** One viscosity $\nu$, one horizon $T$, no forcing, periodic boundaries.
  We learn a *single* operator, not a family indexed by $(\nu, T)$.
- **H3 (ground truth).** Targets are computed by a pseudo-spectral ETDRK2 solver on $N=512$ points
  (viscous part integrated exactly, $2/3$ de-aliasing) and subsampled. We treat them as exact:
  the only error we study is the network's.
- **H4 (splits).** Trajectories are independent, so splitting by sample index is legitimate.
  Train / validation / test atmospheres… *fields* are i.i.d. from the same law: **no distribution
  shift**. Test fields are touched exactly once, at the very end.
- **H5 (normalization).** Inputs and outputs are standardized with mean/std computed **on the
  training set only** and frozen — using test statistics would leak information.
- **H6 (metric).** Success is the **relative $L^2$ error per sample**, $\|\hat y-y\|_2/\|y\|_2$,
  averaged over samples; we also monitor the training loss itself (§4).
- **H7 (spectral content).** Stated and checked in §2: the fields we manipulate are essentially
  band-limited — their energy beyond a few tens of Fourier modes is negligible.

> 🧭 **First time with machine learning?** The whole method in four sentences. (1) We collect pairs
> $(u_0, u_T)$ from H1–H3. (2) A *neural network* is a parametric function $f_\theta$; the
> parameters $\theta$ start random. (3) A *loss* $\mathcal{L}(\theta)$ measures how wrong
> $f_\theta$ is on the training pairs. (4) An *optimizer* changes $\theta$ step by step in the
> direction that decreases $\mathcal{L}$ — that is *training*. Everything else is detail, and the
> details we change on purpose (loss shape, weight decay, learning rate) are the subject of §4.

> 📐 **For the rigorous reader.** Training is empirical risk minimization under H1:
> $\hat\theta = \arg\min_\theta \; \frac1n \sum_{i=1}^n \ell\big(f_\theta(u_0^{(i)}),
> u_T^{(i)}\big)$, an unbiased Monte-Carlo estimate of the population risk
> $R(\theta)=\mathbb E_{u_0}\,\ell(f_\theta(u_0), G^\dagger u_0)$. Generalization claims are
> *always* conditional on H1: the FNO approximates $G^\dagger$ on the support of the GRF, and we
> make no claim about, e.g., discontinuous data. H3 makes "error" well-defined up to solver error,
> which we neglect. Approximation theory for FNOs (Li et al., ICLR 2021): a single FNO layer with all
> modes retained represents any operator whose kernel has an absolutely convergent Fourier series;
> truncating to $m$ modes costs exactly the truncated tail of that series (H7).

In [ ]:
# ---- dataset: three i.i.d. splits, generated on the fly (no download, ~4 s) ----
N_RES   = 64                       # grid points at training time
n_train, n_val, n_test = 128, 32, 32
X, Y, _, _ = load_burgers(n_train=n_train + n_val + n_test, n_test=0,
                          resolution=N_RES, seed=0, cache=False)
Xtr, Ytr = X[:n_train],          Y[:n_train]                      # training: drives the loss
Xva, Yva = X[n_train:-n_test],   Y[n_train:-n_test]               # validation: chooses hyperparams
Xte, Yte = X[-n_test:],          Y[-n_test:]                      # test: untouched until §5
print('splits:', Xtr.shape, Xva.shape, Xte.shape, '(B, C=1, N=64)')

# Normalization (H5): stats from the TRAINING fields only, then frozen.
x_norm, y_norm = ChannelNormalizer(Xtr), ChannelNormalizer(Ytr)
print(f'train  mean={Xtr.mean():+.4f}  std={Xtr.std():.4f}   ->  the network sees ~N(0,1) inputs')

fig, ax = plt.subplots(1, 3, figsize=(11, 3), sharey=True)
x_grid = np.linspace(0, 1, N_RES, endpoint=False)
for i in range(3):
    ax[i].plot(x_grid, Xtr[i, 0], color='0.6', label=r'$u_0$ (input)')
    ax[i].plot(x_grid, Ytr[i, 0], color='C0', lw=2, label=r'$u(\cdot,T)$ (target)')
    ax[i].set_title(f'training sample {i}')
    ax[i].set_xlabel('x');  ax[i].grid(alpha=0.3)
ax[0].set_ylabel('u');  ax[0].legend()
fig.suptitle(r'Input and target fields: the operator $G^\dagger$ steepens and shifts the profile', y=1.04)
plt.show()

In [ ]:
# ---- the statistical distributions of the INPUTS and the OUTPUTS (know your data!) ----
fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.2))
ax[0].hist(Xtr.numpy().ravel(), bins=60, color='C0', alpha=0.8, label='u0 (input)')
ax[0].hist(Ytr.numpy().ravel(), bins=60, color='C3', alpha=0.55, label='u(.,T) (output)')
ax[0].set_xlabel('field value'); ax[0].set_ylabel('count'); ax[0].legend(fontsize=8)
ax[0].set_title('INPUT and OUTPUT value distributions')
ax[1].hist(Xtr.flatten(1).norm(dim=1).numpy(), bins=24, color='C0', alpha=0.8, label='||u0||_2')
ax[1].hist(Ytr.flatten(1).norm(dim=1).numpy(), bins=24, color='C3', alpha=0.7, label='||uT||_2')
ax[1].set_xlabel('per-sample L2 norm'); ax[1].legend(fontsize=8)
ax[1].set_title('per-sample amplitudes (the imbalance of section 4)')
ratio = (Ytr.flatten(1).norm(dim=1) / Xtr.flatten(1).norm(dim=1)).numpy()
ax[2].hist(ratio, bins=24, color='C2', alpha=0.85)
ax[2].set_xlabel('||u(.,T)|| / ||u0||')
ax[2].set_title('how much the operator damps each sample')
plt.tight_layout(); plt.show()
print(f'amplitude spread: ||u0|| varies by a factor {Xtr.flatten(1).norm(dim=1).max() / Xtr.flatten(1).norm(dim=1).min():.1f}; '
      f'the operator maps every input to an output with ~{ratio.mean():.0%} of its norm:')
print('steepening moves energy to high modes, viscosity dissipates it - the distributions show both.')

In [ ]:
# ---- animation 1: the dynamics themselves (why this operator is non-trivial) ----
# We re-solve the PDE at increasing horizons t: watch the nonlinearity steepen the wave.
u0_sel = Xtr[:3, 0].double()                       # same 3 samples as above
t_grid = np.linspace(0, 0.3, 13)                  # 13 frames from t=0 to t=T
frames_u = [solve_burgers(u0_sel, nu=0.008, T=float(t)).numpy() for t in t_grid]

fig, ax = plt.subplots(figsize=(8, 3.4))
colors = ['C0', 'C1', 'C2']
lines = [ax.plot(x_grid, frames_u[0][i], color=colors[i], lw=2)[0] for i in range(3)]
title = ax.set_title('Burgers dynamics, t = 0.000')
ax.set_xlim(0, 1); ax.set_ylim(-3.2, 3.2); ax.set_xlabel('x'); ax.set_ylabel('u'); ax.grid(alpha=0.3)

def update(f):
    for i, ln in enumerate(lines):
        ln.set_ydata(frames_u[f][i])
    title.set_text(f'Burgers dynamics, t = {t_grid[f]:.3f}')
    return lines + [title]

show_anim(fig, update, frames=len(t_grid), interval=400)

> 🧭 **What to notice.** Around $t\approx 0.1$–$0.2$ the steep front forms: the solution *creates
> small scales from large ones*. A network that only looks at a small window of $x$ (a classic
> convolutional net) has to chain many layers to move information across the domain; the FNO's
> Fourier layer moves information across the *whole domain in one step* (§2).
>
> 📐 **Note.** The map $t \mapsto u(\cdot,t)$ above is the true semi-flow; we only learn its
> endpoint $t=T$. Everything here extends to time-stepped operators, but one fixed $T$ keeps the
> hypothesis list short (H2).

## §2 · The one new idea: spectral convolution — 08–14 min · ⏱ checkpoint: start §3 at minute 14

**Convolution theorem, the 20-second version.** For a real signal $v$ on $N$ points, the real FFT
gives $N/2{+}1$ complex coefficients $\hat v_k = \langle v, e^{2\pi i k x}\rangle$ (we use the
`ortho` normalization). A **convolution** in $x$ is a **pointwise multiplication** in Fourier:
$(K v)\hat{\;}_k = R[k]\,\hat v_k$. The FNO layer keeps exactly this form — one learned complex
matrix $R[k]\in\mathbb C^{C_{in}\times C_{out}}$ **per retained mode** $k < m$ — and kills the rest:

$$(K v)\hat{\;}_k = R[k]\, \hat v_k \;\; (k < m), \qquad (K v)\hat{\;}_k = 0 \;\; (k \geq m).$$

- **$m$ (`modes`) is a hypothesis, not a detail**: it says "the operator's action lives in the first
  $m$ frequencies" (H7). Below we *check* H7 on data before trusting it.
- **Any grid size $N$**: the parameters live in frequency space, not on the grid. A model trained on
  $N=64$ can be evaluated on $N=256$ with **zero retraining** (§5 shows it).
- **Globality**: mode $k=1$ connects points a whole domain apart — this is what beats the "many local
  layers" problem of §1.

> 📐 **Kernel view.** A general linear operator on functions has kernel
> $\kappa(x,y)=\sum_{k,k'} \kappa_{k,k'} e^{2\pi i k x} e^{-2\pi i k' y}$; the FNO spectral
> layer is the **diagonal** part $\kappa_{k,k'} \propto \delta_{k,k'} R[k]$, i.e. exactly the
> translation-invariant operators, truncated at $m$. The pointwise $1\times1$ convolution $W$ that
> accompanies $K$ in each block restores the general (local, non-invariant) part of the operator.
> Truncation error $\le$ the $k\ge m$ tail of the kernel's Fourier series — small when the kernel
> is smooth, which is precisely H7.

First, the empirical check of H7 — where does the energy of our data live?

In [ ]:
# ---- H7 checked on data: spectral decay of the targets ----
spec = torch.fft.rfft(Ytr[:, 0].double(), norm='ortho').abs().pow(2).mean(0).numpy()
spec /= spec.sum()
cum = np.cumsum(spec)
m_star = int(np.searchsorted(cum, 0.999)) + 1     # modes needed for 99.9% of the energy

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].semilogy(spec + 1e-12, color='C0')
ax[0].axvline(m_star, color='C3', ls='--', label=f'99.9% of energy: {m_star} modes')
ax[0].set_xlabel('Fourier mode $k$'); ax[0].set_ylabel('mean energy'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[0].set_title('H7 on data: spectra decay fast')
ax[1].plot(cum[:80], color='C0'); ax[1].axhline(0.999, color='C3', ls='--')
ax[1].set_xlabel('modes kept $m$'); ax[1].set_ylabel('cumulative energy'); ax[1].grid(alpha=0.3)
ax[1].set_title('truncating at modest $m$ loses almost nothing')
plt.show()
print(f'Keeping m={m_star} of {len(spec)} modes retains 99.9% of the mean spectral energy.')

In [ ]:
# ---- animation 2: truncating the Fourier series of a solution, mode by mode ----
u = Ytr[0, 0].double()                        # one target field
u_hat = torch.fft.rfft(u, norm='ortho')
mode_list = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
(line,) = ax[0].plot(x_grid, u.numpy(), color='C0', lw=2, label='truth')
(approx,) = ax[0].plot(x_grid, u.numpy(), color='C3', ls='--', lw=1.8, label='truncated')
ax[0].set_ylim(-3.2, 3.2); ax[0].set_xlabel('x'); ax[0].legend(); ax[0].grid(alpha=0.3)
bars = ax[1].bar(np.arange(len(u_hat)), u_hat.abs().numpy(), color='C0')
ax[1].set_yscale('log'); ax[1].set_xlabel('mode $k$'); ax[1].set_ylabel(r'$|\hat u_k|$'); ax[1].grid(alpha=0.3)
title = ax[0].set_title('m = 1')

def update(f):
    m = mode_list[f]
    coef = torch.zeros_like(u_hat); coef[:m] = u_hat[:m]
    rec = torch.fft.irfft(coef, n=N_RES, norm='ortho')
    approx.set_ydata(rec.numpy())
    err = (rec - u).norm() / u.norm()
    title.set_text(f'm = {m} kept modes, rel. error = {err:.3f}')
    for b in bars:                                   # highlight the kept modes in red
        b.set_color('C3' if b.get_x() < m else 'C0')
    return line, approx, title

show_anim(fig, update, frames=len(mode_list), interval=550)

**Exercise F1 · 5 minutes.** Implement the spectral convolution yourself — it is 3 lines inside
`forward`: (1) forward real FFT, (2) multiply the first `modes` coefficients by the learned complex
weight, (3) inverse real FFT (ask for `n=N` so odd grids work too). The cell after yours runs an
automatic check against a reference; if it fails you keep the reference version, and the lab
continues. ⏱ hard stop at minute 14: not passing yet? Read the checker's hint after class and
move on — the fallback keeps the lab on track.

> 🧭 **Why complex weights?** Each $R[k]$ rotates **and** scales each mode's complex coefficient.
> Rotation in Fourier = translation in space: that is how one layer learns shifts and wave motion.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class MySpectralConv1d(nn.Module):
    """(B, Cin, N) -> (B, Cout, N); trainable complex weights, one matrix per Fourier mode."""
    def __init__(self, in_channels, out_channels, modes):
        super().__init__()
        self.in_channels, self.out_channels, self.modes = in_channels, out_channels, modes
        scale = 1.0 / (in_channels * out_channels)
        self.weight = nn.Parameter(
            scale * torch.randn(in_channels, out_channels, modes, dtype=torch.cfloat))

    def forward(self, x):
        B, _, N = x.shape
        x_hat = torch.fft.rfft(x, dim=-1, norm='ortho')      # (B, Cin, N//2+1), complex
        n_modes = min(self.modes, x_hat.shape[-1])
        # TODO 1: allocate a COMPLEX zero tensor of shape (B, Cout, N//2+1)
        #         hint: x_hat.new_zeros(B, self.out_channels, x_hat.shape[-1])
        # TODO 2: fill the first n_modes entries:
        #         out[:, :, :n_modes] = torch.einsum('bik,iok->bok',
        #                                            x_hat[:, :, :n_modes],
        #                                            self.weight[..., :n_modes])
        # TODO 3: return torch.fft.irfft(out, n=N, dim=-1, norm='ortho')
        raise NotImplementedError('Complete the three steps in MySpectralConv1d.forward')

In [ ]:
passed_f1 = check_spectral_conv1d(MySpectralConv1d)      # silent reference comparison
print('F1:', 'your implementation is active' if passed_f1 else 'reference fallback active (finish F1 later)')

# A one-cell demonstration of WHAT the complex weights can do: rotate every mode by 90 degrees.
layer = MySpectralConv1d(1, 1, 8)
with torch.no_grad():
    layer.weight.fill_(1j)                               # R[k] = i for every retained mode k
    xg = torch.arange(N_RES).float() / N_RES
    v = torch.cos(2 * math.pi * 3 * xg)[None, None]      # a pure wave, mode k = 3
    out = layer(v)[0, 0]
    shifted = torch.cos(2 * math.pi * 3 * xg + math.pi / 2)   # quarter-period translation
    print('90-degree rotation in Fourier = quarter-period shift in space:',
          f'max |out - shifted theory| = {(out - shifted).abs().max():.2e}')
    plt.figure(figsize=(5.2, 2.6))
    plt.plot(xg, v[0, 0], color='C0', label='input wave')
    plt.plot(xg, out, color='C3', ls='--', label='after R[k] = i')
    plt.xlabel('x'); plt.legend(); plt.grid(alpha=0.3)
    plt.title(r'rotation in Fourier $\leftrightarrow$ translation in space')
    plt.show()

## §3 · Assemble the FNO and train it — 14–20 min · ⏱ checkpoint: start §4 at minute 20

One spectral layer is a linear, translation-invariant operator. The full FNO alternates it with
pointwise nonlinearities, exactly like a classic CNN alternates convolution with ReLU:

$$a(x) \xrightarrow{\;P\;} v_0 \xrightarrow{\text{block}} v_1 \;\cdots\; v_{L-1}
\xrightarrow{\text{block}} v_L \xrightarrow{\;Q\;} u(x),
\qquad v_{\ell+1} = \mathrm{GELU}\big(\underbrace{K_\ell v_\ell}_{\text{global, } m \text{ modes}}
+ \underbrace{W_\ell v_\ell}_{\text{local, } 1{\times}1}\big)$$

$P$ lifts 1 channel to `width` channels; $Q$ projects back to 1 channel. For training we use the
reference `FNO1d` so that everyone gets identical numbers, whatever happened in F1.

**Why lift to `width` channels at all — where is the computation?** $P$ embeds the physical field
into a learned **latent space**: after lifting, every grid point carries $w = 32$ learned feature
channels, and *the blocks never see $u$ itself again* — all spectral mixing, all nonlinearity,
happens in this learned representation, until $Q$ reads it back to physical units. The width is
the **rank of the representation**: the hypothesis that 32 learned fields per point suffice to
carry the operator's action (more width = richer representation at $O(w^2)$ cost per layer).
Nothing in the architecture says what the 32 channels mean — they are learned features, which is
precisely what makes the model powerful and why we will *look inside* them in Lab 2 §3.5
(there we visualize the analogous learned codes and latent fields directly). Encode → compute
in a latent space → decode is the shared skeleton of every lab in this school.

| hyperparameter | value | why this value (the lab's answer) |
|---|---|---|
| `width` | 32 | enough channels to mix information; 2× wider ≈ 4× cost |
| `modes` $m$ | 16 | above the 99.9%-energy mark of §2 (H7 satisfied with margin) |
| `n_layers` | 4 | depth composes nonlinear spectral mixing; 4 keeps the lab fast |
| loss | relative $L^2$ | the standard of the FNO literature; see §4 for alternatives |
| optimizer | AdamW, lr $2\cdot10^{-3}$ + cosine decay | adaptive step per parameter; §4 shows why |
| `weight_decay` | $10^{-4}$ | mild $L^2$ penalty on real weights — a loss hyperparameter (§4) |
| epochs / batch | 60 / 32 | a teaching-size budget, not a record attempt |

The training loop below is written out **in full, no hidden magic**: normalize (H5), draw a batch,
forward, loss, backward, step. It also snapshots predictions on three validation fields so we can
*watch* the operator being learned.

In [ ]:
def fit_fno(model, *, epochs=60, batch=32, lr=2e-3, wd=1e-4, loss_name='rel_l2',
            snap_every=None, n_snap=3, seed=0):
    """Minimal instrumented training loop (AdamW + cosine schedule). Returns (hist, snaps)."""
    torch.manual_seed(seed)
    loss_fn = get_loss(loss_name)
    decay, no_decay = [], []                       # weight decay only on real 2D weights (H: §4)
    for p in model.parameters():
        (no_decay if (p.is_complex() or p.ndim <= 1) else decay).append(p)
    opt = torch.optim.AdamW([{'params': decay, 'weight_decay': wd},
                             {'params': no_decay, 'weight_decay': 0.0}], lr=lr)
    steps = math.ceil(n_train / batch) * epochs
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)

    xtr, ytr = x_norm.encode(Xtr).to(DEVICE), y_norm.encode(Ytr).to(DEVICE)
    xva = x_norm.encode(Xva).to(DEVICE)
    snap_x = xva[:n_snap]
    model = model.to(DEVICE)
    hist, snaps = {'train': [], 'val': [], 'lr': []}, []
    gen = torch.Generator().manual_seed(seed)
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n_train, generator=gen)
        tot, nb = 0.0, 0
        for i in range(0, n_train, batch):
            idx = perm[i:i + batch]
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(xtr[idx]), ytr[idx])
            loss.backward(); opt.step(); sched.step()
            tot += loss.item(); nb += 1
        model.eval()
        with torch.no_grad():
            pred = y_norm.decode(model(xva)).cpu()          # physical units, H6
        hist['train'].append(tot / nb)
        hist['val'].append(get_loss('rel_l2')(pred, Yva).item())
        hist['lr'].append(sched.get_last_lr()[0])
        if snap_every and (ep % snap_every == 0 or ep == 1):
            with torch.no_grad():
                snaps.append((ep, y_norm.decode(model(snap_x)).cpu().numpy()))
    return hist, snaps

torch.manual_seed(0)
fno = FNO1d(1, 1, width=32, modes=16, n_layers=4)
print(f'FNO1d: {sum(p.numel() * (2 if p.is_complex() else 1) for p in fno.parameters()):,} real parameters '
      '(spectral weights count twice: complex)')
hist, snaps = fit_fno(fno, epochs=60, snap_every=5)
print(f"final validation relative L2 error: {hist['val'][-1]:.4f}  "
      f"(0 = perfect, 1 = predicting zero)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(hist['train'], label='training loss (what the optimizer sees)')
ax.plot(hist['val'], label='validation rel-L2 (what we report, physical units)')
ax.set_xlabel('epoch'); ax.set_ylabel('loss / error'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Both curves fall together: no overfitting to explain at this size')
plt.show()

In [ ]:
# ---- animation 3: the operator being learned, epoch by epoch ----
fig, ax = plt.subplots(1, 4, figsize=(12.5, 3.1))
eps   = [s[0] for s in snaps]
curves = []
for j in range(3):
    truth = Yva[j, 0].numpy()
    ax[j].plot(x_grid, truth, color='C0', lw=2, label='truth')
    (ln,) = ax[j].plot(x_grid, snaps[0][1][j, 0], color='C3', ls='--', lw=1.8, label='FNO')
    ax[j].set_ylim(-3.2, 3.2); ax[j].grid(alpha=0.3); ax[j].set_xlabel('x')
    curves.append(ln)
ax[0].set_ylabel('u'); ax[0].legend()
(loss_ln,) = ax[3].plot(eps, [hist['val'][e - 1] for e in eps], color='0.4')
(dot,) = ax[3].plot([], [], 'o', color='C3')
ax[3].set_xlabel('epoch'); ax[3].set_ylabel('val rel-L2'); ax[3].grid(alpha=0.3)
title = fig.suptitle('', y=1.02)

def update(f):
    ep, preds = snaps[f]
    for j in range(3):
        curves[j].set_ydata(preds[j, 0])
    dot.set_data([ep], [hist['val'][ep - 1]])
    title.set_text(f'epoch {ep}: predictions vs truth on 3 unseen validation fields')

show_anim(fig, update, frames=len(snaps), interval=420)

> 🧭 **What to notice.** In the first frames the network already gets the *location* of the
> structures right (the spectral part is global) but misses the sharp fronts (they live in high
> modes). Watch the fronts sharpen epoch after epoch.
>
> 📐 **What the loop computes.** Each epoch is one pass of stochastic gradient descent over the
> empirical risk of §1. The validation curve is *not* used by the optimizer: it only certifies that
> the training loss and the population-level error (H6) decrease together.

## §4 · The loss, its hyperparameters, and its landscape — 20–26 min · ⏱ checkpoint: start §5 at minute 26

The training loss is the **only** thing the optimizer ever sees. Getting it wrong means training a
perfect optimizer towards the wrong goal. Four standard choices (per sample, then averaged over the
batch; $e = \hat y - y$, $N$ points):

| name | formula per sample | character |
|---|---|---|
| absolute $L^1$ | $\frac1N\sum_x \lvert e(x)\rvert$ | robust to outliers, median-like |
| $L^2$ / MSE | $\frac1N\sum_x e(x)^2$ | smooth, emphasizes large errors |
| relative $L^1$ | $\|e\|_1/\|y\|_1$ | every sample has equal weight |
| relative $L^2$ | $\|e\|_2/\|y\|_2$ | idem; the FNO-paper standard |

**The hyperparameters of the loss** — in the sense of *choices you must make about the objective
before any optimizer runs* — are:

1. **The loss family** (table above). It fixes *which samples matter*. On this dataset
   $\|y\|_2$ varies by a factor ≈ 5 between samples: MSE spends its effort on the loud
   fields, the relative $L^2$ treats quiet and loud fields equally. Both are legitimate goals —
   but they define different experiments, and you must choose (see the sweep results below).
2. **The weight decay $\lambda$** — the loss actually minimized is
   $\mathcal{L}(\theta) + \lambda\|\theta\|_2^2$: a *penalty*, i.e. a prior belief that small
   weights generalize better. (We apply it to real weights only, never to the complex spectral
   weights — decaying those would erase exactly the global transport the FNO needs.)
3. **The learning rate $\eta$** — not part of the loss, but it selects *which trajectory* the
   optimizer takes through the landscape that the loss defines. Too small: crawl. Too large:
   divergence. We will *watch* it below.

### 4.1 · Sweep: loss family × learning rate

Six short trainings (40 epochs each) — identical in every other respect. They take ~2 min on a
laptop CPU and seconds on the cluster GPU: launch the cell and keep reading while it runs.

In [ ]:
# ---- 4.1: what the loss family and the learning rate actually change ----
LOSS_FAMILY = ['l2', 'rel_l2']
LRS = [5e-4, 2e-3, 1e-2]
sweep = {}
t0 = time.time()
for lname in LOSS_FAMILY:
    for lr in LRS:
        torch.manual_seed(0)
        h, _ = fit_fno(FNO1d(1, 1, width=32, modes=16, n_layers=4),
                       epochs=40, lr=lr, loss_name=lname)
        sweep[(lname, lr)] = h
print(f'sweep: {len(sweep)} trainings in {time.time()-t0:.0f} s on {DEVICE}')

for (lname, lr), h in sweep.items():
    print(f"loss = {'MSE' if lname == 'l2' else 'rel-L2':6s}  lr = {lr:.0e}: "
          f"final val rel-L2 = {h['val'][-1]:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for a, lname in zip(ax, LOSS_FAMILY):
    for lr in LRS:
        a.plot(sweep[(lname, lr)]['val'], label=f'lr = {lr:.0e}')
    a.set_title(f"loss = {'MSE' if lname == 'l2' else 'relative L2'}")
    a.set_xlabel('epoch'); a.grid(alpha=0.3); a.legend()
ax[0].set_ylabel('validation rel-L2')
fig.suptitle('Same architecture, same data: only the objective and the step size changed', y=1.03)
plt.show()

amp = Ytr.flatten(1).norm(dim=1)
print(f'amplitude spread: ||y||_2 ranges from {amp.min():.2f} to {amp.max():.2f} '
      f'(ratio {amp.max()/amp.min():.1f}): the RELATIVE loss weighs every sample equally, '
      'MSE spends its effort on the loud ones')

In [ ]:
# ---- 4.2: the weight decay lambda, a genuine part of the objective ----
wds = [0.0, 1e-4, 1e-2]
wd_hist = {}
for wd in wds:
    torch.manual_seed(0)
    h, _ = fit_fno(FNO1d(1, 1, width=32, modes=16, n_layers=4), epochs=40, wd=wd)
    wd_hist[wd] = h

fig, ax = plt.subplots(figsize=(7, 3))
for wd, h in wd_hist.items():
    ax.plot(h['train'], ls='--', color=f'C{wds.index(wd)}', alpha=0.6)
    ax.plot(h['val'], label=f'λ = {wd:.0e}', color=f'C{wds.index(wd)}')
ax.set_xlabel('epoch'); ax.set_ylabel('loss (dashed=train) / val rel-L2'); ax.legend(); ax.grid(alpha=0.3)
ax.set_title('Weight decay: mild λ barely changes this small problem, '
             'strong λ visibly restrains the fit')
plt.show()

### 4.2 · The loss landscape — and the trajectories optimizers take through it

The loss $\mathcal{L}(\theta)$ lives in a ~140,000-dimensional space: we cannot plot it. The
standard, honest compromise (Li et al., NeurIPS 2018): pick **two directions** $d_1, d_2$ around the
trained solution $\theta^\star$ and plot the loss on the 2D plane
$\theta(\alpha,\beta) = \theta^\star + \alpha d_1 + \beta d_2$. We choose:

- $d_1 = (\theta_{\mathrm{init}} - \theta^\star)/\|\theta_{\mathrm{init}} - \theta^\star\|$:
  "the direction training came from";
- $d_2$: a random unit vector: "one unseen direction".

Then we **re-run four optimizers from the same initialization** and project each iterate
$\theta_t$ onto the plane: $\alpha_t = \langle \theta_t - \theta^\star, d_1\rangle$,
$\beta_t = \langle \theta_t - \theta^\star, d_2\rangle$. Every run keeps stepping **until it
actually reaches the trained model's loss $L^\star$** (or exhausts its step budget), so the
animation shows the whole descent, finish line included — first in 2-D, then on the 3-D loss
surface, and finally across architectures with a corner plot of the main hyperparameters.

> 📐 **What is rigorous here.** Every point of the landscape is a *true* loss value of a *true*
> parameter vector (no approximation); the projection of trajectories is exact. The only
> approximation is *interpreting a 2D slice as representative* of the high-dimensional geometry.
> The qualitative conclusions (valley floor, curvature, oscillation) are robust; exact distances are
> not. This is a visualization tool, not a proof technique.

In [ ]:
# ---- 4.3: build the landscape on the plane spanned by (d1, d2) around theta* ----
def flat(m):
    """All parameters as one real vector: complex weights stored as (Re, Im) pairs."""
    parts = []
    for q in m.parameters():
        v = q.detach().reshape(-1)
        parts.append(torch.view_as_real(v).flatten() if v.is_complex() else v.flatten())
    return torch.cat([t.cpu() for t in parts])

def load_flat(m, vec):
    i = 0
    for q in m.parameters():
        n = q.numel()
        if q.is_complex():
            q.data.copy_(torch.view_as_complex(
                vec[i:i + 2 * n].reshape(*q.shape, 2).to(q.device)))
            i += 2 * n
        else:
            q.data.copy_(vec[i:i + n].view_as(q).to(q.device)); i += n

torch.manual_seed(0)
theta_star = flat(fno)                                       # trained solution from §3
theta_init = flat(FNO1d(1, 1, width=32, modes=16, n_layers=4))   # same seed = same init
d1 = theta_init - theta_star; s0 = d1.norm(); d1 = d1 / s0
g = torch.Generator().manual_seed(1)
d2 = torch.randn_like(d1); d2 /= d2.norm()
print(f'||theta_init - theta*|| = {s0:.3f}  (the landscape will span this scale)')

xtr_n = x_norm.encode(Xtr).to(DEVICE); ytr_n = y_norm.encode(Ytr).to(DEVICE)
xb, yb = xtr_n[:32], ytr_n[:32]              # one fixed batch: fast and deterministic
probe = FNO1d(1, 1, width=32, modes=16, n_layers=4).to(DEVICE)
loss_fn = get_loss('rel_l2')

@torch.no_grad()
def plane_loss(a, b):
    load_flat(probe, theta_star + a * d1 + b * d2)
    return loss_fn(probe(xb), yb).item()

G = 21
alphas = torch.linspace(-1.15 * s0, 1.15 * s0, G)
betas  = torch.linspace(-1.15 * s0, 1.15 * s0, G)
Z = np.array([[plane_loss(float(a), float(b)) for b in betas] for a in alphas])
print(f'landscape: {G}x{G} = {G*G} evaluations of the true loss on the plane')

In [ ]:
# ---- 4.4: four optimizers, same start — each one runs TO the final loss value ----
with torch.no_grad():
    L_star = plane_loss(0.0, 0.0)          # the loss of the trained model (section 3) on this batch
print(f'goal of every run: reach the trained model loss  L* = {L_star:.4f}  '
      '(a run "arrives" when it gets within 5% of L*)')

def trajectory(kind, lr, momentum=0.0, max_steps=700, tol=1.05):
    """Full-batch descent from theta_init; stops when the loss reaches tol x L* (or max_steps)."""
    load_flat(probe, theta_init)
    opt = {'sgd':  lambda: torch.optim.SGD(probe.parameters(), lr=lr, momentum=momentum),
           'adam': lambda: torch.optim.Adam(probe.parameters(), lr=lr)}[kind]()
    path, losses = [], []
    for t in range(max_steps):
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(probe(xb), yb)
        loss.backward(); opt.step()
        v = flat(probe)
        path.append((float((v - theta_star) @ d1), float((v - theta_star) @ d2)))
        losses.append(loss.item())
        if losses[-1] <= tol * L_star:
            break
    return np.array(path), np.array(losses)

runs = {'SGD lr 1e-2':       trajectory('sgd', 1e-2),
        'SGD lr 5e-2':       trajectory('sgd', 5e-2),
        'SGD+momentum 5e-2': trajectory('sgd', 5e-2, momentum=0.9),
        'Adam lr 2e-3':      trajectory('adam', 2e-3)}

print(f'{"optimizer":20s} {"steps":>6s} {"final loss":>11s}   verdict')
for name, (path, losses) in runs.items():
    arrived = losses[-1] <= 1.05 * L_star
    print(f'{name:20s} {len(losses):6d} {losses[-1]:11.4f}   '
          + ('ARRIVED at the trained loss' if arrived else 'stopped at max steps, still far above L*'))

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8))
from matplotlib.colors import LogNorm
ext = [float(alphas[0]), float(alphas[-1]), float(betas[0]), float(betas[-1])]
im = ax[0].imshow(Z.T, origin='lower', extent=ext, cmap='viridis', aspect='auto', norm=LogNorm())
pc = ax[0].contour(alphas, betas, Z.T, levels=12, colors='white', linewidths=0.6, alpha=0.7)
ax[0].plot(0, 0, 'r*', ms=14, label=r'$\theta^\star$ (trained §3)')
for (name, (path, _)), c in zip(runs.items(), ['C1', 'C3', 'C2', 'w']):
    ax[0].plot(path[:, 0], path[:, 1], color=c, lw=1.4, label=name)
ax[0].set_xlabel(r'$\alpha$ (direction "from init")'); ax[0].set_ylabel(r'$\beta$ (random direction)')
ax[0].legend(fontsize=8); ax[0].set_title('the landscape, and the paths taken (log color)')
for (name, (_, ls)), c in zip(runs.items(), ['C1', 'C3', 'C2', 'k']):
    ax[1].semilogy(ls, color=c, lw=1.5, label=name)
ax[1].axhline(L_star, color='r', ls='--', lw=1.2, label=r'trained loss $L^*$')
ax[1].set_yscale('log'); ax[1].set_xlabel('step'); ax[1].set_ylabel('true loss along the path')
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3); ax[1].set_title('the same race, loss vs step (goal: the red line)')
plt.tight_layout(); plt.show()

In [ ]:
# ---- animation 4: the full race, from the random start to the final loss value ----
fig, ax = plt.subplots(figsize=(6.8, 5), dpi=80)
ax.contourf(alphas, betas, Z.T, levels=20, cmap='viridis')
ax.contour(alphas, betas, Z.T, levels=10, colors='white', linewidths=0.5, alpha=0.6)
ax.plot(0, 0, 'r*', ms=16, label=r'trained $\theta^\star$ (loss $L^*$)')
ax.plot([s0], [0.0], 'ks', ms=6, label='random init')
trajs = {}
colors_ = {'SGD lr 1e-2': 'C1', 'SGD lr 5e-2': 'C3', 'SGD+momentum 5e-2': 'C2', 'Adam lr 2e-3': 'w'}
for name, c in colors_.items():
    (ln,) = ax.plot([], [], color=c, lw=1.6, label=name)
    (pt,) = ax.plot([], [], 'o', color=c, ms=5)
    trajs[name] = (ln, pt)
max_len = max(len(runs[n][1]) for n in runs)
stride = 7                                   # animate every 7th step (the paths are stored fully)
ax.set_xlabel(r'$\alpha$'); ax.set_ylabel(r'$\beta$'); ax.legend(fontsize=8, loc='upper right')
title = ax.set_title('')

def update(f):
    step = min(f * stride, max_len - 1)
    for name, (ln, pt) in trajs.items():
        path, _ = runs[name]
        k = min(step, len(path) - 1)
        ln.set_data(path[:k + 1, 0], path[:k + 1, 1])
        pt.set_data([path[k, 0]], [path[k, 1]])
    title.set_text(f'step {step + 1}/{max_len} — goal: the star, loss L* = {L_star:.3f}')

show_anim(fig, update, frames=max_len // stride + 1, interval=60)

**Why the trajectories look the way they do — every run chased the same goal, the trained loss
$L^*$:**

- **Adam lr $2\cdot10^{-3}$ (white) ARRIVES** at $L^*$ in well under a hundred steps: it rescales
  every parameter's step by its own gradient history (a diagonal preconditioner). In the FNO,
  complex spectral weights and real pointwise weights live at very different scales, and
  per-parameter scaling is exactly what this geometry rewards — the reason the FNO literature
  defaults to Adam(W).
- **SGD + momentum $5\cdot10^{-2}$ (green) crawls towards the goal** and gets close, but needs
  many more steps: momentum accelerates along consistent descent directions, yet the fixed step
  keeps fighting the valley's anisotropy.
- **SGD lr $5\cdot10^{-2}$ (red) moves — but nowhere near fast enough** to arrive within our step
  budget.
- **SGD lr $1\cdot10^{-2}$ (orange) barely leaves the starting plateau**: at the random
  initialization this network outputs near-zero fields, and the relative-$L^2$ surface there is
  extremely flat for a small fixed step. (Lab 2 shows the same effect in an even stronger form.)

> 📐 **One sentence of theory.** For a locally quadratic loss with curvature $H$, plain GD
> converges only for $\eta < 2/\lambda_{\max}(H)$; momentum changes the effective step to
> $\eta/(1-\mu)$ along straight valleys; Adam approximates diagonal preconditioning by
> $1/\sqrt{\text{EMA}(g^2)}$, i.e. a crude but effective inverse-Hessian diagonal. **"Which
> optimizer works" is a property of the loss geometry, not of the optimizer** — run this little
> race on your own problem before trusting any training curve.

### 4.4 · Multiple minima: the same problem has several good answers

The landscape above is centered on *one* trained model, so it shows one minimum. The loss of a
140k-parameter network is genuinely **non-convex**: train a second copy from a different seed
and it reaches a *different* set of weights with a comparable loss. Plotting the loss on the
plane spanned by (the two trained solutions, one random perpendicular direction) makes the
multi-minimum geometry visible — and running Adam from three different random initializations
shows the practical consequence: **where you start decides which minimum you get**.

> 📐 **Why two minima and not "the" minimum.** Non-convexity means the loss has many stationary
> points; gradient methods converge to the basin of attraction of their initialization. The two
> trained models are two *different* parameter vectors with (almost) the same validation error —
> a property called mode connectivity, and the reason practitioners report results over seeds
> (Lab 2 measures the seed spread explicitly).

In [ ]:
# ---- two trainings, two minima: the landscape on the plane that connects them ----
torch.manual_seed(1)
fno_b = FNO1d(1, 1, width=32, modes=16, n_layers=4)
hist_b, _ = fit_fno(fno_b, epochs=60)
print(f'second training: validation rel-L2 {hist_b["val"][-1]:.4f} '
      f'(the first reached {hist["val"][-1]:.4f}) — different weights, comparable loss')

theta_a, theta_b = flat(fno), flat(fno_b)
d_ab = theta_a - theta_b; s_ab = float(d_ab.norm()); d_ab = d_ab / d_ab.norm()
d_perp = torch.randn_like(d_ab)
d_perp = d_perp - (d_perp @ d_ab) * d_ab; d_perp = d_perp / d_perp.norm()
mid = (theta_a + theta_b) / 2

@torch.no_grad()
def plane_loss_mm(a, b):
    load_flat(probe, mid + a * d_ab + b * d_perp)
    return loss_fn(probe(xb), yb).item()

Gm = 17
als = torch.linspace(-0.75 * s_ab, 0.75 * s_ab, Gm)
bes = torch.linspace(-0.75 * s_ab, 0.75 * s_ab, Gm)
Zm = np.array([[plane_loss_mm(float(a), float(b)) for b in bes] for a in als])
print(f'two-minimum landscape: {Gm}x{Gm} evaluations; the trained models sit at '
      f'+/- {s_ab / 2:.1f} along the horizontal axis')

def adam_from(theta0, steps=250, lr=2e-3):
    load_flat(probe, theta0)
    opt = torch.optim.Adam(probe.parameters(), lr=lr)
    path, losses = [], []
    for _ in range(steps):
        opt.zero_grad(set_to_none=True)
        loss = loss_fn(probe(xb), yb)
        loss.backward(); opt.step()
        v = flat(probe)
        path.append((float((v - mid) @ d_ab), float((v - mid) @ d_perp)))
        losses.append(loss.item())
    return np.array(path), np.array(losses)

runs_mm = {}
for sd in (10, 11, 12):
    torch.manual_seed(sd)
    runs_mm[f'random init {sd - 9}'] = adam_from(flat(FNO1d(1, 1, width=32, modes=16, n_layers=4)))
for name, (_, ls_) in runs_mm.items():
    print(f'{name:14s}: loss {ls_[0]:.3f} -> {ls_[-1]:.4f}')

In [ ]:
# ---- animation: three initializations, three basins ----
from matplotlib.colors import LogNorm
fig, ax = plt.subplots(figsize=(7, 5.4))
ext_mm = [float(als[0]), float(als[-1]), float(bes[0]), float(bes[-1])]
ax.imshow(Zm.T, origin='lower', extent=ext_mm, cmap='viridis', aspect='auto', norm=LogNorm())
ax.contour(als, bes, Zm.T, levels=10, colors='white', linewidths=0.5, alpha=0.6)
ax.plot([s_ab / 2], [0], 'r*', ms=17, label='trained model A')
ax.plot([-s_ab / 2], [0], '*', color='C1', ms=17, label='trained model B')
mm_trajs = {}
for (name, (path, _)), c in zip(runs_mm.items(), ['w', 'C2', 'C4']):
    (ln,) = ax.plot([], [], color=c, lw=1.7, label=name)
    (pt,) = ax.plot([], [], 'o', color=c, ms=5)
    mm_trajs[name] = (ln, pt)
ax.set_xlabel('direction connecting the two trained models')
ax.set_ylabel('random perpendicular direction')
ax.legend(fontsize=8, loc='upper right')
title = ax.set_title('')

def update(f):
    for name, (ln, pt) in mm_trajs.items():
        path, _ = runs_mm[name]
        k = min(f, len(path) - 1)
        ln.set_data(path[:k + 1, 0], path[:k + 1, 1])
        pt.set_data([path[k, 0]], [path[k, 1]])
    title.set_text(f'Adam step {f + 1}/250: each initialization falls into its own basin')

show_anim(fig, update, frames=250, interval=45)

**Reading the two-minimum landscape.** The two red/orange stars are the two *trained* networks:
same problem, same architecture, different seeds — different weights, nearly the same validation
error, sitting in **separate minima separated by a loss barrier**. The three Adam runs start
from random initializations and each falls into a different basin. Nothing about this is
pathological: it is the normal geometry of non-convex losses, and it has two honest
consequences. (i) "the trained model" does not exist — report results over seeds. (ii) A single
training run measures one basin, not the method: comparing architectures on one seed each can
rank them wrongly (Lab 2 quantifies the spread: ~14% across five seeds on its task).

### 4.5 · 🥾 Optional deep dive: the same landscape in 3D

The 2-D slice is a projection; the third axis — the loss itself — is what the optimizers actually
descend. Below: the true loss surface $\mathcal{L}(\alpha, \beta)$ over the plane, with the four
trajectories drawn *on* it, and the trained minimum $\theta^\star$ at the bottom.

In [ ]:
# ---- 4.5: the landscape and the trajectories in 3D ----
Zlog = np.log10(np.maximum(Z, 1e-4))              # log loss: the valley walls are steep
Agrid, Bgrid = np.meshgrid(alphas, betas)         # Z[a, b] -> surface over (alpha, beta)

fig = plt.figure(figsize=(9.5, 6.5))
ax3 = fig.add_subplot(projection='3d')
surf = ax3.plot_surface(Agrid, Bgrid, Zlog.T, cmap='viridis', alpha=0.75,
                        linewidth=0, antialiased=True, rstride=1, cstride=1)
for (name, (path, ls)), c in zip(runs.items(), ['C1', 'C3', 'C2', 'w']):
    ax3.plot(path[:, 0], path[:, 1], np.log10(np.maximum(ls, 1e-4)),
             color=c, lw=1.8, label=name)
    ax3.plot([path[-1, 0]], [path[-1, 1]], [np.log10(max(ls[-1], 1e-4))],
             'o', color=c, ms=5)
ax3.scatter([0], [0], [np.log10(max(L_star, 1e-4))], color='r', marker='*', s=160,
             label=r'trained $\theta^\star$ (loss $L^*$)')
ax3.set_xlabel(r'$\alpha$ (from init)'); ax3.set_ylabel(r'$\beta$ (random)')
ax3.set_zlabel(r'$\log_{10}$ loss')
ax3.view_init(elev=28, azim=-60)
ax3.legend(fontsize=8, loc='upper left')
ax3.set_title('the loss landscape with the optimizer trajectories, in 3D')
plt.show()
print(f'Adam (white) dives to the red star; SGD 1e-2 (orange) stays on the high plateau.')

In [ ]:
# ---- animation 5: rotate the 3D landscape (the projection direction is a choice!) ----
fig = plt.figure(figsize=(7.2, 5.8), dpi=80)
ax3 = fig.add_subplot(projection='3d')
ax3.plot_surface(Agrid, Bgrid, Zlog.T, cmap='viridis', alpha=0.75,
                 linewidth=0, antialiased=False)
for (name, (path, ls)), c in zip(runs.items(), ['C1', 'C3', 'C2', 'w']):
    ax3.plot(path[:, 0], path[:, 1], np.log10(np.maximum(ls, 1e-4)), color=c, lw=1.8)
ax3.scatter([0], [0], [np.log10(max(L_star, 1e-4))], color='r', marker='*', s=160)
ax3.set_xlabel(r'$\alpha$'); ax3.set_ylabel(r'$\beta$')
ax3.set_zlabel(r'$\log_{10}$ loss')

def update(f):
    ax3.view_init(elev=28, azim=-75 + f * 5)
    ax3.set_title(f'3D loss landscape, azimuth {-75 + f * 5} degrees')

show_anim(fig, update, frames=30, interval=120)

> 🧭 **What the 3D view adds.** In the flat 2-D projection, a trajectory is just a curve; here you
> see it *descending*. The white Adam line dives into the valley and lands on the red star; the
> orange SGD line wanders on the high plateau. Rotating also shows honestly that a 2-D slice is one
> projection among many — the shape of the "landscape" depends on the plane you chose.
>
> 📐 **Method note.** The surface is the exact loss on the plane
> $\theta^\star + \alpha d_1 + \beta d_2$ (every point a true forward pass), and each
> trajectory's height is its exact loss. Nothing is interpolated or extrapolated across
> directions.

### 4.6 · 🥾 Optional deep dive: the hyperparameter landscape (corner plot)

So far we moved *within* one architecture. The other half of "hyperparameters" lives at the
coarser scale: **the shape of the network itself**. For the FNO1d the main ones are:

- `modes` $m$ — how much of the Fourier spectrum the operator may use (§2);
- `width` $w$ — the channel count, which is **also the width of the lifting map $P$**
  ($P$ is a $1\times1$ convolution into $w$ channels);
- `n_layers` $L$ — how many spectral blocks are composed;
- `proj_hidden` — the width of the projection head $Q$ (the pointwise MLP that maps $w$ channels
  back to the solution).

A **corner plot** (as used in astronomy for posterior marginals) shows every 1-D marginal on the
diagonal and every 2-D pair below it, colored by the achieved validation error: it exposes
interactions (e.g. "many modes only help when the width is there to mix them") at a glance.

We run a small random search: 20 configurations sampled from a grid, 25 epochs each — a few
minutes on CPU, seconds on a GPU. (Set the environment variable `CORNER_RUNS` to change the
budget before opening the notebook.)

In [ ]:
# ---- 4.6: random search over (modes, width, n_layers, proj_hidden) + corner plot ----
GRID = {'modes':       [4, 8, 16, 24, 32],
        'width':       [8, 16, 32, 48],         # also the width of the lifting map P
        'n_layers':    [2, 3, 4, 6],
        'proj_hidden': [16, 32, 64, 128]}       # the width of the projection head Q
rng = np.random.default_rng(7)
N_CORNER = int(os.environ.get('CORNER_RUNS', '20'))
seen, cfgs = set(), []
while len(cfgs) < N_CORNER:
    cfg = {k: int(rng.choice(v)) for k, v in GRID.items()}
    key = tuple(cfg.values())
    if key not in seen:
        seen.add(key); cfgs.append(cfg)

print(f'{N_CORNER} configurations x 25 epochs '
      f'(~5 min on a laptop CPU, seconds on a GPU):')
corner_loss = []
for i, cfg in enumerate(cfgs):
    torch.manual_seed(0)
    h, _ = fit_fno(FNO1d(1, 1, **cfg), epochs=25)
    corner_loss.append(h['val'][-1])
    if (i + 1) % 5 == 0 or i == 0:
        print(f'  [{i + 1:2d}/{N_CORNER}] {cfg} -> val rel-L2 {h["val"][-1]:.4f}')
corner_loss = np.array(corner_loss)
best = int(corner_loss.argmin())
print('best configuration:', cfgs[best], '-> val rel-L2', round(float(corner_loss[best]), 4))

In [ ]:
# ---- the corner plot: 1-D marginals on the diagonal, 2-D pairs below, color = error ----
NAMES = list(GRID)
LBL = {'modes': 'modes m', 'width': 'width w (P width)', 'n_layers': 'layers L',
       'proj_hidden': 'proj_hidden (Q width)'}
fig, axes = plt.subplots(4, 4, figsize=(12, 10.5))
sc = None
for r in range(4):
    for c in range(4):
        ax = axes[r, c]
        xr = np.array([cfg[NAMES[r]] for cfg in cfgs])
        xc = np.array([cfg[NAMES[c]] for cfg in cfgs])
        if r == c:
            ax.scatter(xr, corner_loss, s=26, c='C0')
            ax.scatter(xr[best], corner_loss[best], marker='*', s=200, c='C3', zorder=5)
            ax.set_ylabel('val rel-L2'); ax.set_xlabel(LBL[NAMES[r]])
            ax.grid(alpha=0.3)
        elif r > c:
            sc = ax.scatter(xc, xr, c=corner_loss, cmap='viridis', s=42,
                            edgecolor='k', lw=0.3)
            ax.scatter(xc[best], xr[best], marker='*', s=220, c='C3', zorder=5,
                       edgecolor='k', lw=0.4)
            ax.set_xlabel(LBL[NAMES[c]]); ax.set_ylabel(LBL[NAMES[r]])
        else:
            ax.axis('off')
fig.colorbar(sc, ax=axes, shrink=0.55, label='validation rel-L2 (dark = better)')
fig.suptitle('Hyperparameter corner plot: 20 architectures, 25 epochs each '
             '(red star = best of the search)', y=0.97)
plt.show()

**Reading the corner plot — and its honest limits.**

- **Diagonals.** The validation error falls as `modes` grows towards the spectral mark of §2, then
  saturates: beyond the data's actual spectral content, extra modes buy nothing. `width` and
  `proj_hidden` show the classic pattern: too small starves the operator, then a plateau —
  this problem is small and does not need a large network.
- **Off-diagonal panels.** Look for *tilted* point clouds: e.g. configurations with many modes
  but tiny width (top-left corner of the (width, modes) panel) sit high — modes without channel
  capacity to mix them are wasted parameters. That is an **interaction**: no 1-D marginal can
  show it.
- **Honest limits.** (i) Each point is a 25-epoch budget: larger architectures are penalized
  (they converge slower), so the corner plot ranks *at this budget*, not at convergence.
  (ii) 20 points is a coarse scan of a 4-D space: it reveals trends, not optima — the starred
  best-of-search is a good starting point, not a proof. (iii) The loss on the vertical scale is
  the *validation* error: the plot shows what generalizes, not what fits the training set.

## §5 · Evaluate, super-resolve, transfer — 26–30 min · ⏱ wrap-up by minute 30

Now — and only now — the **test set** (H4). We report all four losses of §4's table, we look at the
worst sample (a scalar score hides local failures), and we run the experiment that no
grid-fixed-network can do: evaluate the *same* model, trained on $N=64$, **directly at $N=256$
without any retraining**.

In [ ]:
# ---- final evaluation on the untouched test split ----
fno.eval()
with torch.no_grad():
    pred = y_norm.decode(fno(x_norm.encode(Xte).to(DEVICE))).cpu()
print('test metrics:', {k: round(v, 4) for k, v in all_metrics(pred, Yte).items()})

rel = (pred.flatten(1) - Yte.flatten(1)).norm(dim=1) / Yte.flatten(1).norm(dim=1)
i_best, i_worst = int(rel.argmin()), int(rel.argmax())
fig, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for a, i, t in zip(ax, [i_best, i_worst], ['best', 'worst']):
    a.plot(x_grid, Yte[i, 0], color='C0', lw=2, label='truth')
    a.plot(x_grid, pred[i, 0], color='C3', ls='--', lw=1.8, label='FNO')
    a.set_title(f'{t} test sample (rel-L2 = {rel[i]:.3f})'); a.set_xlabel('x'); a.grid(alpha=0.3)
ax[0].legend(); ax[0].set_ylabel('u')
plt.show()
print(f'test rel-L2: mean {rel.mean():.4f} | median {rel.median():.4f} | worst {rel.max():.4f}')

In [ ]:
# ---- zero-shot super-resolution: same model, finer grid, no retraining ----
N_HI = 256
Xte_hi, Yte_hi, _, _ = load_burgers(n_train=n_test, n_test=0, resolution=N_HI, seed=5, cache=False)
with torch.no_grad():
    pred_hi = y_norm.decode(fno(x_norm.encode(Xte_hi).to(DEVICE))).cpu()
rel_hi = (pred_hi.flatten(1) - Yte_hi.flatten(1)).norm(dim=1) / Yte_hi.flatten(1).norm(dim=1)

fig, ax = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
x_hi = np.linspace(0, 1, N_HI, endpoint=False)
ax[0].plot(x_grid, Yte[0, 0], color='C0', lw=2, label='truth')
ax[0].plot(x_grid, pred[0, 0], color='C3', ls='--', lw=1.6, label='FNO')
ax[0].set_title('test sample, N = 64 (training resolution)'); ax[0].grid(alpha=0.3)
ax[1].plot(x_hi, Yte_hi[0, 0], color='C0', lw=2, label='truth')
ax[1].plot(x_hi, pred_hi[0, 0], color='C3', ls='--', lw=1.6, label='FNO')
ax[1].set_title('another test sample, N = 256 (never seen)'); ax[1].grid(alpha=0.3)
for a in ax: a.set_xlabel('x')
ax[0].set_ylabel('u'); ax[0].legend()
plt.show()
print(f'test rel-L2 at N=64: {rel.mean():.4f}   |   at N=256 (zero-shot): {rel_hi.mean():.4f}')

The error at $N=256$ is of the same order as at $N=64$: the parameters are attached to **Fourier
modes, not to grid points** (§2), so resolution is an evaluation detail, not an architectural one.
This is the operational difference between an FNO and an "ML surrogate of a fixed grid".

## Takeaways — and using FNOs on *your* problem

- **What we did.** Learned $G^\dagger: u_0 \mapsto u(\cdot,T)$ for Burgers from 128 examples,
  with every assumption stated (H1–H7) and checked where checkable (H7).
- **Why spectral.** The FFT makes the layer global ($O(N\log N)$), resolution-agnostic, and its
  truncation is an explicit, testable smoothness hypothesis — not a magic number.
- **What the loss did.** §4 is the heart of the lab: the loss family and $\lambda$ define *the
  goal*; the learning rate only chooses the path through the landscape the goal defines. Watch the
  landscape animation once more before believing any "optimizer X is better" claim.

**Checklist: FNO on your application.**
1. Write your operator $G^\dagger$ explicitly: what is the input function, on which domain, and
   what physical parameter(s) index the family you want (H2: fix them or feed them as channels).
2. Generate (or collect) i.i.d. pairs from **one** distribution; split by sample (H4).
3. Check H7 yourself: plot the mean spectral energy of your targets; set `modes` above the
   99.9%-energy mark.
4. Normalize from training statistics only (H5).
5. Start with `width=32, modes≈spectral mark, n_layers=4`, AdamW lr $2\cdot10^{-3}$, cosine decay,
   relative $L^2$ — then re-run §4's sweeps for *your* data: amplitudes there are usually less
   homogeneous than here.
6. Evaluate in physical units on held-out data (H6), look at worst cases, and test the
   resolution-invariance claim on *your* grids.
7. If your domain is not rectangular/periodic: pad, or move to **Lab 3** (graphs handle irregular
   geometry natively).

*Next: Lab 2 applies the factorized variant (FFNO) to infrasound propagation in the atmospheric
boundary layer; Lab 3 replaces the Fourier grid by an irregular graph.*

### Optional — save this interactive session measurement (Hedy only)
On the cluster, the next cell writes a JSON report with the elapsed interactive time of this lab
(the data the instructors use to pace the school). On a laptop it prints a one-line summary.

## References

**Operator learning and FNO/FFNO:**
- Z. Li, N. Kovachki, K. Azizzadenesheli, B. Liu, K. Bhattacharya, A. Stuart, A. Anandkumar.
  *Fourier Neural Operator for Parametric Partial Differential Equations.* ICLR 2021.
- Z. Li, N. Kovachki, ... A. Anandkumar. *Neural Operator: Learning Maps Between Function
  Spaces.* Nature Machine Intelligence **6**, 666–676 (2024).
- A. Tran, A. Mathews, L. Xie, C.-K. Ong. *Factorized Fourier Neural Operators.* ICLR 2023.
- K. Azizzadenesheli, Z. Li, ... A. Anandkumar. *Neural Operators for Accelerating Scientific
  Simulations? Open Problems and Perspectives.* ICLR 2024 (position paper).

**Graph neural networks:**
- T. N. Kipf, M. Welling. *Semi-Supervised Classification with Graph Convolutional Networks.*
  ICLR 2017.
- J. Gilmer, S. S. Schoenholz, P. F. Riley, O. Vinyals, G. E. Dahl. *Neural Message Passing for
  Quantum Chemistry.* NeurIPS 2017.
- Q. Li, Z. Han, X.-M. Wu. *Deeper Insights into Graph Convolutional Networks for Semi-
  Supervised Learning.* AAAI 2018. (Oversmoothing and the Dirichlet-energy analysis.)
- R. L. Murphy, B. Srinivasan, V. Rao, B. Ribeiro. *Janossy Pooling: Learning Deep
  Permutation-Invariant Functions.* NeurIPS 2019. (Why permutation-invariant aggregation
  matters.)

**Loss landscapes and optimization:**
- H. Li, Z. Xu, G. Taylor, C. Studer, T. Goldstein. *Visualizing the Loss Landscape of Neural
  Nets.* NeurIPS 2018. (The 2-D random-slice methodology used in Lab 1 §4.)
- T. Garipov, P. Izmailov, D. Podoprikhin, D. Vetrov, A. Wilson. *Loss Surfaces, Mode
  Connectivity, and Fast Ensembling of DNNs.* NeurIPS 2018. (Why two trained networks are
  connected by a low-loss path — the multi-minimum geometry of Lab 1 §4.4.)

**Infrasound propagation and parabolic-equation methods:**
- A. Le Pichon, E. Blanc, A. Hauchecorne (eds.). *Infrasound Monitoring for Atmospheric
  Studies.* 2nd ed., Springer (2021).
- R. Waxler, J. Assink. *Propagation of infrasound through the atmosphere: theory and
  observations.* In: Infrasound Monitoring, ch. 5 (2019).
- D. P. Knupp, S. J. Franke. *A global reference atmosphere for infrasound propagation.*
  J. Acoust. Soc. Am. **140**, 2724 (2016).

**Software:**
- A. Paszke et al. *PyTorch: An Imperative Style, High-Performance Deep Learning Library.*
  NeurIPS 2019.
- M. Fey, J. E. Lenssen. *Fast Graph Representation Learning with PyTorch Geometric.*
  ICLR Workshop on Representation Learning on Graphs and Manifolds (2019).


In [ ]:
if HEDY_SESSION is not None:
    HEDY_SESSION.report()   # writes a JSON timing report on the Hedy cluster only
else:
    print('No Hedy session: nothing to report (laptop run).')